# MMD Domain Adaptation Runner

Run the strict recording-disjoint MMD crash test directly from this notebook. The training cell below uses the full schedule by default.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_root = cwd.parents[1] if cwd.name == 'mmd_domain_adaptation' else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from crash_tests.mmd_domain_adaptation.config import MMDDomainAdaptationConfig
from crash_tests.mmd_domain_adaptation.run_experiment import run_with_config


## Config

In [2]:
RUN_NAME = 'mmd_notebook_full'
RUN_FINETUNE_STAGE = True

SOURCE_TRAIN_EPOCHS = 25
FINETUNE_EPOCHS = 12
LAMBDA_GRID = (0.0, 0.01, 0.05, 0.1, 0.25)

config = MMDDomainAdaptationConfig(
    parquet_path=repo_root / 'data/raw/33000_ROWS.parquet',
    output_root=repo_root / 'crash_tests' / 'mmd_domain_adaptation' / 'outputs',
    run_name=RUN_NAME,
    run_finetune_stage=RUN_FINETUNE_STAGE,
    source_train_epochs=SOURCE_TRAIN_EPOCHS,
    finetune_epochs=FINETUNE_EPOCHS,
    lambda_grid=LAMBDA_GRID,
    verbose=True,
)

display(pd.DataFrame([config.to_dict()]))


,parquet_path,output_root,run_name,random_state,paired_final_holdout_rows,targets,source_fmiss_target,lambda_grid,mmd_kernel,mmd_bandwidth_multipliers,...,source_lr,finetune_lr,weight_decay,patience,lightgbm_estimators,lightgbm_learning_rate,lightgbm_num_leaves,save_checkpoints,verbose,resolved_output_dir
0,/Users/paulruiz/Documents/Predicting_Good_Unit...,/Users/paulruiz/Documents/Predicting_Good_Unit...,mmd_notebook_full,42,100,"(fpos, fmiss)",fmiss_extended,"(0.0, 0.01, 0.05, 0.1, 0.25)",rbf_multi_scale,"(0.25, 0.5, 1.0, 2.0, 4.0)",...,0.002,0.0008,0.0001,5,250,0.05,31,True,True,/Users/paulruiz/Documents/Predicting_Good_Unit...


## Run Training

Execute the next cell to run the experiment. The cell output will stream the source task loss, MMD loss, and finetune loss.

In [3]:
output_dir = run_with_config(config)
print(f'Saved outputs to {output_dir}')
output_dir


Loading parquet from /Users/paulruiz/Documents/Predicting_Good_Units/data/raw/33000_ROWS.parquet
Running target=fpos
fpos:lambda=0.00:cv1:source: lambda=0.0000 source_rows=20376 target_unlabeled_rows=2721 epochs=25
fpos:lambda=0.00:cv1:source epoch=1/25 task=2.264492 mmd=0.176095 total=2.264492 val_mae=0.064225
fpos:lambda=0.00:cv1:source epoch=2/25 task=1.393575 mmd=0.170846 total=1.393575 val_mae=0.059363
fpos:lambda=0.00:cv1:source epoch=3/25 task=1.198124 mmd=0.154571 total=1.198124 val_mae=0.047877
fpos:lambda=0.00:cv1:source epoch=4/25 task=1.103001 mmd=0.153234 total=1.103001 val_mae=0.049271
fpos:lambda=0.00:cv1:source epoch=5/25 task=1.049981 mmd=0.167977 total=1.049981 val_mae=0.047966
fpos:lambda=0.00:cv1:source epoch=6/25 task=0.984781 mmd=0.163897 total=0.984781 val_mae=0.046566
fpos:lambda=0.00:cv1:source epoch=7/25 task=0.963193 mmd=0.156961 total=0.963193 val_mae=0.045240
fpos:lambda=0.00:cv1:source epoch=8/25 task=0.926416 mmd=0.134429 total=0.926416 val_mae=0.041802
f

PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/crash_tests/mmd_domain_adaptation/outputs/mmd_notebook_full')

## Review Outputs

In [6]:
metrics = pd.read_csv(output_dir / 'metrics.csv')
baselines = pd.read_csv(output_dir / 'baseline_comparison.csv')
lambda_sweep = pd.read_csv(output_dir / 'lambda_sweep.csv')
history = pd.read_csv(output_dir / 'training_history.csv')
split_manifest = json.loads((output_dir / 'split_manifest.json').read_text())

display(pd.DataFrame([split_manifest['main']]))
display(metrics.sort_values(['target', 'mae', 'variant_id']))


,allow_test_time_unlabeled_context,context_baseline_feature_count,holdout_recordings,hybrid_source_rows,paired_finetune_recordings,paired_finetune_rows,paired_non_test_rows,paired_test_recordings,paired_test_rows,protocol,shape_feature_count,ssl_pool_recordings,ssl_pool_rows,training_side_scaler_rows,unit_feature_count
0,True,685,"[PAIRED_BOYDEN::paired_boyden32c::1103_1_1, PA...",24211,14,107,5070,15,100,recording_disjoint,70,16,4963,29281,164


,protocol,target,variant_id,family,model_id,source_target,eval_stage,lambda,selected_lambda,mae,rmse,r2,bias,calibration_slope,calibration_intercept
0,main,fmiss,contextual_lightgbm_no_study_id,baseline,contextual_lightgbm_no_study_id,NaN,holdout,NaN,NaN,0.186768,0.231416,0.361718,0.044327,1.044117,-0.058526
1,main,fmiss,neural_no_mmd_finetuned,neural_no_mmd,neural_no_mmd_finetuned,fmiss_extended,holdout,0.00,0.00,0.256911,0.291845,-0.015153,0.100980,0.590957,0.053843
2,main,fmiss,paired_only_lightgbm_shape,baseline,paired_only_lightgbm_shape,NaN,holdout,NaN,NaN,0.260319,0.317302,-0.199978,0.022013,0.124732,0.240160
3,main,fmiss,neural_mmd_finetuned,neural_mmd,neural_mmd_finetuned,fmiss_extended,holdout,0.10,0.10,0.260643,0.309531,-0.141917,0.055433,0.380829,0.150723
4,main,fmiss,neural_mmd_zero_shot,neural_mmd,neural_mmd_zero_shot,fmiss_extended,holdout,0.10,0.10,0.278363,0.347059,-0.435602,0.070058,0.299254,0.173507
5,main,fmiss,neural_no_mmd_zero_shot,neural_no_mmd,neural_no_mmd_zero_shot,fmiss_extended,holdout,0.00,0.00,0.468186,0.539576,-2.470026,0.302907,0.062435,0.241283
6,main,fpos,neural_mmd_finetuned,neural_mmd,neural_mmd_finetuned,fpos,holdout,0.25,0.25,0.150319,0.205458,0.304869,0.027606,0.712287,0.043895
7,main,fpos,contextual_lightgbm_no_study_id,baseline,contextual_lightgbm_no_study_id,NaN,holdout,NaN,NaN,0.153214,0.196654,0.363164,0.022210,0.898287,0.002518
8,main,fpos,paired_only_lightgbm_shape,baseline,paired_only_lightgbm_shape,NaN,holdout,NaN,NaN,0.159735,0.202304,0.326047,0.036435,0.864874,-0.001661
9,main,fpos,neural_no_mmd_finetuned,neural_no_mmd,neural_no_mmd_finetuned,fpos,holdout,0.00,0.00,0.207306,0.287251,-0.358763,0.030266,0.298003,0.146059


In [5]:
print('Lambda sweep')
display(lambda_sweep.sort_values(['target', 'eval_stage', 'cv_mae_mean', 'lambda']))

print('Training history tail')
display(history.tail(60))

print('Baseline comparison')
display(baselines.sort_values(['target', 'mae', 'variant_id']))


Lambda sweep


,target,eval_stage,lambda,cv_mae_mean,cv_rmse_mean,cv_r2_mean,cv_bias_mean,cv_calibration_slope_mean,cv_calibration_intercept_mean,n_splits
0,fmiss,finetuned,0.10,0.256454,0.331285,-0.447184,-0.079303,0.261661,0.201687,3
1,fmiss,finetuned,0.01,0.263119,0.357275,-0.739985,-0.003371,0.327286,0.251935,3
2,fmiss,finetuned,0.25,0.263446,0.331763,-0.436183,-0.067505,0.080682,0.263683,3
3,fmiss,finetuned,0.05,0.297850,0.382712,-1.007669,-0.095989,0.250020,0.263506,3
4,fmiss,finetuned,0.00,0.328860,0.402155,-1.281931,0.216521,0.091417,0.238390,3
5,fmiss,zero_shot,0.10,0.292363,0.395122,-1.098213,0.034397,0.080038,0.263749,3
6,fmiss,zero_shot,0.01,0.336943,0.430470,-1.533241,0.141432,0.047018,0.303630,3
7,fmiss,zero_shot,0.25,0.364017,0.462560,-1.865047,0.212250,0.110637,0.258637,3
8,fmiss,zero_shot,0.05,0.378678,0.488353,-2.113715,0.210571,0.032913,0.277852,3
9,fmiss,zero_shot,0.00,0.563253,0.648321,-4.724246,0.553957,0.208595,0.103929,3


Training history tail


,protocol,target,variant_id,eval_stage,lambda,stage,split_id,epoch,train_task_loss,train_mmd_loss,train_total_loss,train_loss,val_mae
1228,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,15,0.358472,0.114546,0.358472,NaN,0.042433
1229,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,16,0.354256,0.128412,0.354256,NaN,0.041195
1230,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,17,0.357091,0.158666,0.357091,NaN,0.043789
1231,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,18,0.344842,0.119634,0.344842,NaN,0.038700
1232,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,19,0.327738,0.109943,0.327738,NaN,0.041694
1233,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,20,0.325531,0.108318,0.325531,NaN,0.039401
1234,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,21,0.331603,0.100751,0.331603,NaN,0.040138
1235,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,22,0.320039,0.119000,0.320039,NaN,0.044145
1236,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,23,0.306285,0.117426,0.306285,NaN,0.037961
1237,main,fmiss,neural_no_mmd_finetuned,finetuned,0.0,source,NaN,24,0.308960,0.114966,0.308960,NaN,0.039533


Baseline comparison


,protocol,target,variant_id,family,model_id,source_target,eval_stage,lambda,selected_lambda,mae,rmse,r2,bias,calibration_slope,calibration_intercept
0,main,fmiss,contextual_lightgbm_no_study_id,baseline,contextual_lightgbm_no_study_id,NaN,holdout,NaN,NaN,0.186768,0.231416,0.361718,0.044327,1.044117,-0.058526
1,main,fmiss,neural_no_mmd_finetuned,neural_no_mmd,neural_no_mmd_finetuned,fmiss_extended,holdout,0.00,0.00,0.256911,0.291845,-0.015153,0.100980,0.590957,0.053843
2,main,fmiss,paired_only_lightgbm_shape,baseline,paired_only_lightgbm_shape,NaN,holdout,NaN,NaN,0.260319,0.317302,-0.199978,0.022013,0.124732,0.240160
3,main,fmiss,neural_mmd_finetuned,neural_mmd,neural_mmd_finetuned,fmiss_extended,holdout,0.10,0.10,0.260643,0.309531,-0.141917,0.055433,0.380829,0.150723
4,main,fmiss,neural_mmd_zero_shot,neural_mmd,neural_mmd_zero_shot,fmiss_extended,holdout,0.10,0.10,0.278363,0.347059,-0.435602,0.070058,0.299254,0.173507
5,main,fmiss,neural_no_mmd_zero_shot,neural_no_mmd,neural_no_mmd_zero_shot,fmiss_extended,holdout,0.00,0.00,0.468186,0.539576,-2.470026,0.302907,0.062435,0.241283
6,main,fpos,neural_mmd_finetuned,neural_mmd,neural_mmd_finetuned,fpos,holdout,0.25,0.25,0.150319,0.205458,0.304869,0.027606,0.712287,0.043895
7,main,fpos,contextual_lightgbm_no_study_id,baseline,contextual_lightgbm_no_study_id,NaN,holdout,NaN,NaN,0.153214,0.196654,0.363164,0.022210,0.898287,0.002518
8,main,fpos,paired_only_lightgbm_shape,baseline,paired_only_lightgbm_shape,NaN,holdout,NaN,NaN,0.159735,0.202304,0.326047,0.036435,0.864874,-0.001661
9,main,fpos,neural_no_mmd_finetuned,neural_no_mmd,neural_no_mmd_finetuned,fpos,holdout,0.00,0.00,0.207306,0.287251,-0.358763,0.030266,0.298003,0.146059
